[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-1_%EC%97%B4%ED%99%94%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%99%80_%EA%B1%B4%EC%A0%84%EC%84%B1%EC%A7%80%ED%91%9C.ipynb)

# 3-1 · 필터가 막혀 가는 과정 — 열화 데이터와 건전성 지표

**⏱ 80분** · 모듈 3: 열화 지표 → **③ 잔여수명(RUL) 예측** → 교체 시점 결정

## 🎯 이번 시간의 질문

> 필터가 **완전히 막히기까지 얼마나 남았는지** 알 수 있을까요?

모듈 2는 "지금 어떤 상태인가"를 판단했습니다. 정비를 **계획**하려면 "앞으로 얼마나 버틸까", 즉 **잔여수명(RUL, Remaining Useful Life)**이 필요합니다.
RUL을 예측하려면 먼저 설비가 **나빠지는 정도를 숫자로 보여 주는 지표**가 있어야 합니다. 이것을 **건전성 지표(HI, Health Indicator)**라고 합니다.
이번 시간에는 실제 고장 실험 데이터에서 HI를 만들고, RUL의 정답을 계산하고, 열화가 **처음 감지되는 시점(P)**부터 **고장(F)**까지의 시간을 봅니다.

## 데이터: 압출기 필터 막힘 실험 (호주 멜버른대, 2026 공개)

플라스틱을 녹여 밀어내는 **압출기**에는 이물질을 걸러내는 금속망 **필터**가 있습니다. 연구팀은 플라스틱에 가루(탄산칼슘)를 섞어 필터가 빨리 막히게 만들고, 고장까지 센서를 기록했습니다.
필터가 막힐수록 플라스틱을 밀어내기 힘들어져 **모터 전류가 올라갑니다.** 처리량이 멈춘 시점이 **고장(F)**입니다.

| 열 | 뜻 |
|---|---|
| `run_id` | 실험 번호 (필터 한 개 = 실험 한 번) |
| `time_s` | 실험 시작 후 경과 시간(초). 10초마다 한 행 (10초 평균값) |
| `current_a` | 모터 전류(A) — 이번 모듈의 핵심 센서 |
| `voltage_v`, `feed_c` ~ `front_c` | 전압, 구간별 온도 |
| `failure_s` | 이 실험이 고장 난 시간 F(초) |
| `split` | 학습용(train) 33회 / 평가용(test) 8회 |

막힘 실험 42회 중 기록 시간이 맞지 않는 1회를 뺀 **41회**를 씁니다(파열 고장 18회는 다른 고장이라 제외).

**이번 시간에 익힐 패턴:** 한 실험 골라 그리기 · 열 계산(`F − t`) · 이동평균 `rolling` · `groupby(...).last()` · 조건을 만족하는 첫 시점 찾기

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
runs = pd.read_csv('data/filter_runs.csv')
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()      # 분석 대상 41회
print('실험 수:', main['run_id'].nunique(), '/ 행 수:', len(main))
main.head()

## 1. 필터 한 개의 일생 보기

학습용 실험 중 **24번 필터**(F = 425초)를 골라 전류 변화를 그립니다. `plt.axvline`으로 고장 시점에 세로선을 긋습니다.
(`.copy()`는 골라낸 표를 따로 복사해 두는 것입니다. 뒤에서 새 열을 추가할 때 필요합니다.)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one = main[main['run_id'] == 24].copy()
F = one['failure_s'].iloc[0]
plt.plot(one['time_s'], one['current_a'], marker='.')
plt.axvline(F, color='red', linestyle='--', label=f'고장 F = {F}초')
plt.xlabel('경과 시간 (초)')
plt.ylabel('모터 전류 (A)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 처음 150초 정도는 0.6 A 근처에서 평평하다가, 이후 전류가 꾸준히 올라 고장 직전 약 1.15 A가 됩니다.
필터에 이물질이 쌓이는 **열화 과정이 전류에 그대로 드러납니다.** 이런 센서값이 건전성 지표(HI)의 후보입니다.

## 2. RUL의 정답 계산하기

고장 시간 F를 알고 있으므로, 각 시점 t의 **실제 잔여수명 = F − t** 입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one['RUL'] = one['failure_s'] - one['time_s']
plt.plot(one['time_s'], one['RUL'])
plt.xlabel('경과 시간 (초)')
plt.ylabel('실제 잔여수명 RUL (초)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 RUL은 시간이 갈수록 **일직선으로** 줄어 고장 시점에 0이 됩니다. 이것이 모델이 맞혀야 할 **정답**입니다.
⚠️ 이 정답은 **고장이 난 뒤에야** 계산할 수 있습니다. 현장에서 실시간으로 쓰는 필터는 F를 모르므로, 센서만 보고 RUL을 **추정**해야 합니다(3-2, 3-3).
데이터에는 같은 값이 `RUL_s` 열로 이미 들어 있습니다.

## 3. 건전성 지표(HI) 만들기 — 잡음 줄이기

10초 평균 전류도 오르내림(잡음)이 있습니다. **이동평균**으로 최근 3개(30초) 값을 평균하면 잡음이 줄어듭니다.
`rolling(3, min_periods=1).mean()` = "최근 3개씩 평균, 처음처럼 값이 3개가 안 되면 있는 만큼만 평균".

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one['hi'] = one['current_a'].rolling(3, min_periods=1).mean()
plt.plot(one['time_s'], one['current_a'], alpha=0.5, label='10초 평균 전류')
plt.plot(one['time_s'], one['hi'], linewidth=2, label='HI (최근 30초 이동평균)')
plt.xlabel('경과 시간 (초)')
plt.ylabel('전류 (A)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 HI(주황)는 흔들림이 줄어 **추세가 더 잘 보입니다.** 대신 과거 값을 평균하므로 변화를 조금 **늦게** 따라갑니다. 잡음 제거와 반응 속도는 교환 관계입니다(과제 2).

### ▶ 41회 실험 전체에 HI 만들기

같은 계산을 모든 실험에 적용합니다. `groupby('run_id')`로 **실험마다 따로** 이동평균을 계산해야 앞 실험의 값이 다음 실험에 섞이지 않습니다.

In [ ]:
# ▶ 그대로 실행하세요
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
train = main[main['split'] == 'train']
test = main[main['split'] == 'test']

plt.figure(figsize=(9, 4))
for rid, run in train.groupby('run_id'):
    plt.plot(run['time_s'] - run['failure_s'], run['hi'], alpha=0.5)
plt.xlabel('고장까지 남은 시간의 반대 (t − F, 초) · 0 = 고장')
plt.ylabel('HI (A)')
plt.title('학습용 33개 필터의 HI — 고장 시점(0)에 맞춰 정렬')
plt.xlim(-800, 0)
plt.show()

💬 **결과 해설** — 모든 필터가 고장(0)을 향해 HI가 올라가는 **공통 경향**이 있습니다. 하지만
- 오르기 시작하는 시점과 속도가 제각각이고,
- 어떤 필터는 일찍 올라간 뒤 **오랫동안 평평하게** 버티다 고장 나며,
- 고장 순간의 HI도 0.84~1.18 A로 다릅니다.

"HI가 몇 A가 되면 고장"이라는 **고장 기준선이 하나로 딱 정해지지 않는다**는 뜻입니다. 얼마나 흩어져 있는지 숫자로 확인합니다.

## 4. 고장 기준선 — 고장 순간의 HI는 얼마인가

**P3** — 학습용 실험마다 마지막 행(고장 직전)의 HI를 `groupby(...).last()`로 뽑습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
end_hi = train.groupby('run_id')['hi'].last()
print(end_hi.describe().round(3))
plt.hist(end_hi, bins=10)
plt.xlabel('고장 직전 HI (A)')
plt.ylabel('실험 수')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 중앙값(50%) 1.069 A, 최소 0.835 A, 최대 1.177 A.

💬 **결과 해설** — 절반의 필터는 1.07 A보다 낮은 HI에서 고장 났습니다. 기준선을 1.07 A로 잡으면 **절반은 기준선에 닿기 전에 고장** 납니다. 기준선을 낮게(보수적으로) 잡을수록 안전하지만 일찍 교체하게 됩니다.
다음 시간(3-2)에는 이 기준선(중앙값 1.07 A)을 이용해 "HI가 언제 기준선에 닿을까"로 RUL을 추정합니다.

## 5. 열화 감지 시점 P와 P–F 구간

1-2에서 배운 **P–F 구간**을 여기서도 계산합니다. 규칙: **"HI가 처음 30초 평균보다 0.1 A 이상 올라가면 열화 감지(P)"**.
이것은 2-2의 이상탐지와 같은 원리입니다. 필터 **자신의 초기 상태**를 정상 기준으로 삼는 것이 다를 뿐입니다.

**조건을 만족하는 첫 시점 찾기** — 조건으로 행을 고른 뒤(`P2`) 가장 이른 시간(`min()`)을 구합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
base = one['current_a'].iloc[:3].mean()          # 처음 3개 행(30초)의 평균 = 이 필터의 정상 기준
P = one.loc[one['hi'] > base + 0.1, 'time_s'].min()
print('초기 기준:', round(base, 3), 'A')
print('감지 시점 P:', P, '초 / 고장 F:', F, '초 / P–F:', F - P, '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 24번 필터는 **240초**에 열화가 감지되었고, 고장까지 **185초**의 여유가 있었습니다. 이 185초 안에 교체를 준비해야 합니다.

### ▶ 학습용 33개 필터 전체의 P–F 구간

같은 규칙을 모든 학습용 필터에 적용하는 함수입니다(`margin` = 초기 기준보다 얼마나 올라가면 감지할지).

In [ ]:
# ▶ 그대로 실행하세요
def detect_P(df, margin=0.1):
    """필터마다 열화 감지 시점 P, 고장 F, P–F 구간을 표로 만든다. 감지 못 하면 P는 빈칸(NaN)."""
    rows = []
    for rid, run in df.groupby('run_id'):
        base = run['current_a'].iloc[:3].mean()
        P = run.loc[run['hi'] > base + margin, 'time_s'].min()
        F = run['failure_s'].iloc[0]
        rows.append({'run_id': rid, 'P': P, 'F': F, 'PF': F - P})
    return pd.DataFrame(rows)

pf = detect_P(train, margin=0.1)
print('감지한 필터:', pf['P'].notna().sum(), '/', len(pf))
print('P–F 중앙값:', pf['PF'].median(), '초 / 최소:', pf['PF'].min(), '초')
plt.hist(pf['PF'], bins=12)
plt.xlabel('P–F 구간 (초)')
plt.ylabel('필터 수')
plt.show()

💬 **결과 해설** — 33개 모두 고장 전에 열화를 감지했고, P–F 구간의 중앙값은 **310초**입니다. 하지만 가장 짧은 필터는 **29초**뿐이었습니다(57번: 110초에 감지, 139초에 고장).
**P–F 구간도 필터마다 크게 다릅니다.** "감지되면 교체"라는 규칙만으로는 준비 시간이 부족한 필터가 생깁니다 → "감지 후 **얼마나 남았는지**"를 추정하는 RUL 예측이 필요한 이유입니다.

---
## 6. 실습 과제 (30분)

### 📝 과제 1 · 감지 기준(margin)을 바꾸면?
`detect_P(train, margin=...)`의 margin을 **0.05, 0.1, 0.2, 0.3**으로 바꿔 표를 채우세요.

| margin (A) | 감지한 필터 수 | P–F 중앙값(초) | P–F 최소(초) |
|---|---|---|---|
| 0.05 | | | |
| 0.1 | 33 | 310 | 29 |
| 0.2 | | | |
| 0.3 | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for m in [0.05, 0.1, 0.2, 0.3]: pf_m = detect_P(train, margin=m) 후 위와 같이 출력
# (본문의 pf는 margin 0.1 결과로 과제 3에서 다시 쓰므로 덮어쓰지 않습니다.)


✏️ **나의 답:** margin을 키우면 어떻게 되나요? 너무 작으면 어떤 문제가 생길까요?

### 📝 과제 2 · 이동평균 창 크기의 영향
24번 필터에서 이동평균 창을 **1(평균 안 함), 3, 6**으로 바꿔 한 그림에 겹쳐 그리세요. 창을 키우면 무엇을 얻고 무엇을 잃나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for k in [1, 3, 6]: plt.plot(one['time_s'], one['current_a'].rolling(k, min_periods=1).mean(), label=f'창 {k}')


✏️ **나의 답:**

### 📝 과제 3 · 준비 시간이 120초 필요하다면?
필터 교체 준비(작업자 호출, 예비 필터 준비)에 **120초**가 걸린다고 합시다. margin = 0.1일 때, 학습용 33개 중 **P–F가 120초 이상인** 필터는 몇 개인가요? 나머지 필터는 어떻게 될까요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: (pf['PF'] >= 120).sum()


✏️ **나의 답:**

### 📝 과제 4 · 생각해 보기 — 전류를 HI로 쓸 때의 장단점
모터 전류를 필터의 건전성 지표로 쓸 때의 **장점 한 가지**와 **주의할 점 한 가지**를 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| RUL 정답 | RUL = F − t. 고장 후에야 알 수 있는 값 → 실시간으로는 **추정**해야 한다 |
| 건전성 지표 HI | 열화를 보여 주는 센서값. 여기서는 전류의 30초 이동평균 |
| 이동평균 | 잡음↓ 대신 반응 지연↑ |
| 고장 기준선 | 고장 순간의 HI가 필터마다 달라(0.84~1.18 A) 하나의 값으로 정하기 어렵다 |
| P–F 구간 | 감지 후 고장까지. 중앙값 310초지만 최소 29초 → **감지만으로는 부족, 남은 시간 추정 필요** |

다음: **3-2 · RUL 예측 ① — 추세를 연장해서 고장선에 닿는 시간 구하기**